In [2]:
!pip install sacrebleu datasets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.1 MB/s eta 0:00:00


In [3]:
# ============================================================
# Bidirectional Weight Sharing for Language Translation
#
# Exp 1a: Custom LSTM seq2seq — EN→DE (separate enc/dec cells)
# Exp 1b: Custom LSTM seq2seq — DE→EN (separate enc/dec cells)
# Exp  2: One shared LSTM cell — handles both EN→DE and DE→EN
#
# Dataset : Multi30k EN↔DE
# Metric  : BLEU (sacrebleu)
# ============================================================
#
# Run these in Colab/Kaggle before executing:
#   !pip install datasets sacrebleu --quiet
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import sacrebleu
import random, time, json

# ── CONFIG ───────────────────────────────────────────────────
CONFIG = {
    'embed_dim' : 256,
    'hidden_dim': 512,
    'dropout'   : 0.3,
    'batch_size': 128,
    'lr'        : 1e-3,
    'n_epochs'  : 20,
    'patience'  : 5,
    'clip'      : 1.0,
    'tf_ratio'  : 0.5,
    'min_freq'  : 2,
    'max_len'   : 50,
    'seed'      : 42,
    'save_path' : '/kaggle/working/nmt_results.json',
    'device'    : 'cuda' if torch.cuda.is_available() else 'cpu',
}

torch.manual_seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
print(f"Device: {CONFIG['device']}")

PAD, SOS, EOS, UNK = 0, 1, 2, 3


# ── VOCABULARY ───────────────────────────────────────────────
class Vocab:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.w2i = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
        self.i2w = {0: '<pad>', 1: '<sos>', 2: '<eos>', 3: '<unk>'}

    def build(self, sentences):
        cnt = Counter(w for s in sentences for w in s.lower().split())
        for w, f in cnt.items():
            if f >= self.min_freq and w not in self.w2i:
                i = len(self.w2i)
                self.w2i[w] = i
                self.i2w[i] = w

    def encode(self, s, max_len=None):
        toks = s.lower().split()
        if max_len:
            toks = toks[:max_len]
        return [SOS] + [self.w2i.get(t, UNK) for t in toks] + [EOS]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS:
                break
            if i not in (PAD, SOS):
                out.append(self.i2w.get(i, '<unk>'))
        return ' '.join(out)

    def __len__(self):
        return len(self.w2i)


# ── DATA LOADING ─────────────────────────────────────────────
def load_data():
    print("\nLoading Multi30k from HuggingFace...")
    try:
        from datasets import load_dataset
        ds = load_dataset("bentrevett/multi30k")
        tr_en = [x['en'] for x in ds['train']]
        tr_de = [x['de'] for x in ds['train']]
        va_en = [x['en'] for x in ds['validation']]
        va_de = [x['de'] for x in ds['validation']]
        te_en = [x['en'] for x in ds['test']]
        te_de = [x['de'] for x in ds['test']]
    except Exception as e:
        raise RuntimeError(f"Could not load Multi30k: {e}\n"
                           "Run: !pip install datasets")

    print(f"Train {len(tr_en)} | Val {len(va_en)} | Test {len(te_en)}")

    en_vocab = Vocab(CONFIG['min_freq'])
    de_vocab = Vocab(CONFIG['min_freq'])
    en_vocab.build(tr_en)
    de_vocab.build(tr_de)
    print(f"EN vocab {len(en_vocab):,} | DE vocab {len(de_vocab):,}")

    return tr_en, tr_de, va_en, va_de, te_en, te_de, en_vocab, de_vocab


class NMTDataset(Dataset):
    def __init__(self, src, tgt, sv, tv, ml=50):
        self.data = [
            (torch.tensor(sv.encode(s, ml)),
             torch.tensor(tv.encode(t, ml)))
            for s, t in zip(src, tgt)
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]


def collate(batch):
    s, t = zip(*batch)
    return (pad_sequence(s, batch_first=True, padding_value=PAD),
            pad_sequence(t, batch_first=True, padding_value=PAD))


def make_loaders(tr_en, tr_de, va_en, va_de, te_en, te_de,
                 en_vocab, de_vocab):
    ML, BS = CONFIG['max_len'], CONFIG['batch_size']

    def dl(src, tgt, sv, tv, shuffle):
        return DataLoader(NMTDataset(src, tgt, sv, tv, ML),
                          BS, shuffle=shuffle, collate_fn=collate)

    en2de = {
        'train': dl(tr_en, tr_de, en_vocab, de_vocab, True),
        'val':   dl(va_en, va_de, en_vocab, de_vocab, False),
        'test':  dl(te_en, te_de, en_vocab, de_vocab, False),
    }
    de2en = {
        'train': dl(tr_de, tr_en, de_vocab, en_vocab, True),
        'val':   dl(va_de, va_en, de_vocab, en_vocab, False),
        'test':  dl(te_de, te_en, de_vocab, en_vocab, False),
    }
    print(f"EN→DE train batches: {len(en2de['train'])} | "
          f"DE→EN train batches: {len(de2en['train'])}")
    return en2de, de2en


# ── LSTM CELL (Adigun-style ParameterDict) ───────────────────
class LSTMCell(nn.Module):
    """
    Custom LSTM cell with explicit ParameterDict.
    Direction-blind: the cell has no notion of forward vs backward,
    source vs target, or English vs German. Those roles live in the
    pipeline that calls it.
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        s = 0.01
        self.p = nn.ParameterDict({
            'Wii': nn.Parameter(torch.randn(hidden_size, input_size)  * s),
            'Whi': nn.Parameter(torch.randn(hidden_size, hidden_size) * s),
            'bi' : nn.Parameter(torch.zeros(hidden_size)),
            'Wif': nn.Parameter(torch.randn(hidden_size, input_size)  * s),
            'Whf': nn.Parameter(torch.randn(hidden_size, hidden_size) * s),
            'bf' : nn.Parameter(torch.zeros(hidden_size)),
            'Wig': nn.Parameter(torch.randn(hidden_size, input_size)  * s),
            'Whg': nn.Parameter(torch.randn(hidden_size, hidden_size) * s),
            'bg' : nn.Parameter(torch.zeros(hidden_size)),
            'Wio': nn.Parameter(torch.randn(hidden_size, input_size)  * s),
            'Who': nn.Parameter(torch.randn(hidden_size, hidden_size) * s),
            'bo' : nn.Parameter(torch.zeros(hidden_size)),
        })

    def forward(self, x, h, c):
        p = self.p
        i = torch.sigmoid(x @ p['Wii'].T + h @ p['Whi'].T + p['bi'])
        f = torch.sigmoid(x @ p['Wif'].T + h @ p['Whf'].T + p['bf'])
        g = torch.tanh(   x @ p['Wig'].T + h @ p['Whg'].T + p['bg'])
        o = torch.sigmoid(x @ p['Wio'].T + h @ p['Who'].T + p['bo'])
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new


# ── EXP 1: STANDARD SEQ2SEQ (separate encoder + decoder cells)
class Exp1Seq2Seq(nn.Module):
    """
    Standard seq2seq with two separate LSTM cells:
      enc_cell — encodes source sequence
      dec_cell — decodes into target tokens
    enc_cell IS NOT dec_cell.
    One model handles one translation direction only.
    """
    def __init__(self, src_vsz, tgt_vsz, emb_dim, hid_dim, drop=0.3):
        super().__init__()
        self.emb_src  = nn.Embedding(src_vsz, emb_dim, padding_idx=PAD)
        self.emb_tgt  = nn.Embedding(tgt_vsz, emb_dim, padding_idx=PAD)
        self.enc_cell = LSTMCell(emb_dim, hid_dim)   # encoder weights
        self.dec_cell = LSTMCell(emb_dim, hid_dim)   # decoder weights — SEPARATE
        self.fc       = nn.Linear(hid_dim, tgt_vsz)
        self.drop     = nn.Dropout(drop)
        self.hid_dim  = hid_dim

    def _encode(self, src):
        B, dev = src.size(0), src.device
        x = self.drop(self.emb_src(src))
        h = torch.zeros(B, self.hid_dim, device=dev)
        c = torch.zeros(B, self.hid_dim, device=dev)
        for t in range(x.size(1)):
            h, c = self.enc_cell(x[:, t], h, c)
        return h, c

    def forward(self, src, tgt, tf_ratio=0.5):
        B, T = tgt.shape
        V    = self.fc.out_features
        dev  = src.device
        h, c = self._encode(src)
        out  = torch.zeros(B, T, V, device=dev)
        tok  = tgt[:, 0]
        for t in range(1, T):
            e = self.drop(self.emb_tgt(tok))
            h, c = self.dec_cell(e, h, c)
            pred = self.fc(h)
            out[:, t] = pred
            tok = tgt[:, t] if random.random() < tf_ratio else pred.argmax(1)
        return out

    def translate(self, src, max_len=50):
        self.eval()
        dev = src.device
        with torch.no_grad():
            h, c = self._encode(src)
            tok  = torch.full((src.size(0),), SOS, device=dev)
            preds = []
            for _ in range(max_len):
                e = self.emb_tgt(tok)
                h, c = self.dec_cell(e, h, c)
                tok = self.fc(h).argmax(1)
                preds.append(tok)
                if (tok == EOS).all():
                    break
        return torch.stack(preds, dim=1)


# ── EXP 2: SHARED CELL SEQ2SEQ ───────────────────────────────
class Exp2SharedSeq2Seq(nn.Module):
    """
    One LSTMCell shared across ALL four roles:
      1. Encoder for EN→DE
      2. Decoder for EN→DE
      3. Encoder for DE→EN
      4. Decoder for DE→EN

    The cell is direction-blind, role-blind, and language-blind.
    Direction and role are determined by the pipeline.
    Language is handled by separate embedding matrices.

    LSTM params: 1x  (vs 4x for two standard Exp1 models combined)
    """
    def __init__(self, en_vsz, de_vsz, emb_dim, hid_dim, drop=0.3):
        super().__init__()
        # Language-specific embeddings (language identity lives here)
        self.emb_en = nn.Embedding(en_vsz, emb_dim, padding_idx=PAD)
        self.emb_de = nn.Embedding(de_vsz, emb_dim, padding_idx=PAD)

        # ── THE SHARED CELL ──────────────────────────────────
        # Same object. Same memory address. All four roles above
        # call self.cell — there is no other LSTM cell in this model.
        self.cell   = LSTMCell(emb_dim, hid_dim)
        # ─────────────────────────────────────────────────────

        # Language-specific output projections
        self.fc_de  = nn.Linear(hid_dim, de_vsz)  # predicts German tokens
        self.fc_en  = nn.Linear(hid_dim, en_vsz)  # predicts English tokens

        self.drop   = nn.Dropout(drop)
        self.hid_dim = hid_dim

    def _encode(self, emb, dev):
        """Run embedded sequence through the shared cell."""
        B = emb.size(0)
        h = torch.zeros(B, self.hid_dim, device=dev)
        c = torch.zeros(B, self.hid_dim, device=dev)
        for t in range(emb.size(1)):
            h, c = self.cell(emb[:, t], h, c)   # shared cell — encoder role
        return h, c

    def forward(self, src, tgt, direction='en2de', tf_ratio=0.5):
        dev = src.device
        B, T = tgt.shape

        # Select language-specific components based on direction
        if direction == 'en2de':
            src_emb    = self.drop(self.emb_en(src))
            tgt_emb_fn = lambda tok: self.drop(self.emb_de(tok))
            fc         = self.fc_de
        else:  # de2en
            src_emb    = self.drop(self.emb_de(src))
            tgt_emb_fn = lambda tok: self.drop(self.emb_en(tok))
            fc         = self.fc_en

        h, c = self._encode(src_emb, dev)

        V   = fc.out_features
        out = torch.zeros(B, T, V, device=dev)
        tok = tgt[:, 0]

        for t in range(1, T):
            e = tgt_emb_fn(tok)
            h, c = self.cell(e, h, c)           # shared cell — decoder role
            pred = fc(h)
            out[:, t] = pred
            tok = tgt[:, t] if random.random() < tf_ratio else pred.argmax(1)

        return out

    def translate(self, src, direction='en2de', max_len=50):
        self.eval()
        dev = src.device
        with torch.no_grad():
            if direction == 'en2de':
                src_emb    = self.emb_en(src)
                tgt_emb_fn = self.emb_de
                fc         = self.fc_de
            else:
                src_emb    = self.emb_de(src)
                tgt_emb_fn = self.emb_en
                fc         = self.fc_en

            h, c  = self._encode(src_emb, dev)
            tok   = torch.full((src.size(0),), SOS, device=dev)
            preds = []

            for _ in range(max_len):
                e = tgt_emb_fn(tok)
                h, c = self.cell(e, h, c)
                tok = fc(h).argmax(1)
                preds.append(tok)
                if (tok == EOS).all():
                    break

        return torch.stack(preds, dim=1)


# ── TRAINING ─────────────────────────────────────────────────
def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


criterion = nn.CrossEntropyLoss(ignore_index=PAD)


def train_exp1(model, tr_dl, va_dl, name):
    """Training loop for a single-direction Exp1 model."""
    dev  = CONFIG['device']
    opt  = optim.Adam(model.parameters(), lr=CONFIG['lr'])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    best_val = 1e9; patience = 0; best_st = None

    print(f"\n{'='*55}")
    print(f"Training : {name}")
    print(f"Params   : {count_params(model):,}")
    print(f"{'='*55}")

    for ep in range(1, CONFIG['n_epochs'] + 1):
        t0 = time.time()

        # train
        model.train(); tr = 0.0
        for src, tgt in tr_dl:
            src, tgt = src.to(dev), tgt.to(dev)
            opt.zero_grad()
            out  = model(src, tgt, tf_ratio=CONFIG['tf_ratio'])
            loss = criterion(out[:, 1:].reshape(-1, out.size(-1)),
                             tgt[:, 1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG['clip'])
            opt.step()
            tr += loss.item()
        tr /= len(tr_dl)

        # val
        model.eval(); va = 0.0
        with torch.no_grad():
            for src, tgt in va_dl:
                src, tgt = src.to(dev), tgt.to(dev)
                out  = model(src, tgt, tf_ratio=0.0)
                va  += criterion(out[:, 1:].reshape(-1, out.size(-1)),
                                 tgt[:, 1:].reshape(-1)).item()
        va /= len(va_dl)
        sched.step(va)

        print(f"Epoch {ep:02d} | train {tr:.4f} | val {va:.4f} | {time.time()-t0:.1f}s")

        if va < best_val:
            best_val = va; patience = 0
            best_st = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= CONFIG['patience']:
                print(f"Early stop at epoch {ep}")
                break

    model.load_state_dict(best_st)
    return model


def train_exp2(model, en2de, de2en, name):
    """
    Training loop for shared model.
    Alternates EN→DE and DE→EN batches each iteration.
    Gradient from both directions accumulates into the same cell.
    """
    dev   = CONFIG['device']
    opt   = optim.Adam(model.parameters(), lr=CONFIG['lr'])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    best_val = 1e9; patience = 0; best_st = None

    print(f"\n{'='*55}")
    print(f"Training : {name}")
    print(f"Params   : {count_params(model):,}")
    print(f"{'='*55}")

    for ep in range(1, CONFIG['n_epochs'] + 1):
        t0 = time.time()

        # train — alternate directions
        model.train(); tr = 0.0; n = 0
        for (se, td), (sd, te) in zip(en2de['train'], de2en['train']):
            for src, tgt, dirn in [(se, td, 'en2de'), (sd, te, 'de2en')]:
                src, tgt = src.to(dev), tgt.to(dev)
                opt.zero_grad()
                out  = model(src, tgt, direction=dirn, tf_ratio=CONFIG['tf_ratio'])
                loss = criterion(out[:, 1:].reshape(-1, out.size(-1)),
                                 tgt[:, 1:].reshape(-1))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG['clip'])
                opt.step()
                tr += loss.item(); n += 1
        tr /= n

        # val — both directions
        model.eval(); va = 0.0; n = 0
        with torch.no_grad():
            for (se, td), (sd, te) in zip(en2de['val'], de2en['val']):
                for src, tgt, dirn in [(se, td, 'en2de'), (sd, te, 'de2en')]:
                    src, tgt = src.to(dev), tgt.to(dev)
                    out  = model(src, tgt, direction=dirn, tf_ratio=0.0)
                    va  += criterion(out[:, 1:].reshape(-1, out.size(-1)),
                                     tgt[:, 1:].reshape(-1)).item()
                    n   += 1
        va /= n
        sched.step(va)

        print(f"Epoch {ep:02d} | train {tr:.4f} | val {va:.4f} | {time.time()-t0:.1f}s")

        if va < best_val:
            best_val = va; patience = 0
            best_st = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= CONFIG['patience']:
                print(f"Early stop at epoch {ep}")
                break

    model.load_state_dict(best_st)
    return model


# ── EVALUATION ───────────────────────────────────────────────
def compute_bleu(model, loader, tgt_vocab, dev, direction=None, max_len=50):
    model.eval()
    hyps, refs = [], []
    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(dev)
            if direction:
                trans = model.translate(src, direction=direction, max_len=max_len)
            else:
                trans = model.translate(src, max_len=max_len)
            for i in range(src.size(0)):
                hyps.append(tgt_vocab.decode(trans[i].tolist()))
                refs.append(tgt_vocab.decode(tgt[i].tolist()))
    return sacrebleu.corpus_bleu(hyps, [refs]).score


def show_translations(model, src_sents, src_vocab, tgt_vocab, dev,
                      direction=None, n=5, label=""):
    """Print a few example translations for qualitative inspection."""
    ML = CONFIG['max_len']
    print(f"\n── Sample Translations: {label} ──")
    model.eval()
    with torch.no_grad():
        for sent in src_sents[:n]:
            ids = torch.tensor(src_vocab.encode(sent, ML)).unsqueeze(0).to(dev)
            if direction:
                out = model.translate(ids, direction=direction)
            else:
                out = model.translate(ids)
            print(f"  SRC : {sent}")
            print(f"  TGT : {tgt_vocab.decode(out[0].tolist())}")
            print()


# ── WEIGHT SHARING VERIFICATION ─────────────────────────────
def verify_sharing():
    print("\n── Weight Sharing Verification ──")

    # Exp 1: enc and dec must be different objects
    m1 = Exp1Seq2Seq(10, 10, 32, 64)
    assert m1.enc_cell is not m1.dec_cell, "Exp1 sharing check failed"
    print(f"  Exp1 enc_cell IS dec_cell          : {m1.enc_cell is m1.dec_cell}  ✓ (should be False)")

    # Exp 2: one cell used in all four paths
    m2 = Exp2SharedSeq2Seq(10, 10, 32, 64)
    # Confirm there is exactly ONE LSTMCell in the model
    cells = [m for m in m2.modules() if isinstance(m, LSTMCell)]
    assert len(cells) == 1, f"Expected 1 LSTMCell, found {len(cells)}"
    print(f"  Exp2 LSTMCell count in model       : {len(cells)}  ✓ (should be 1)")
    print(f"  Exp2 cell id (encoder = decoder)   : {id(m2.cell)}  ✓ (same object all paths)")

    del m1, m2


# ── MAIN ─────────────────────────────────────────────────────
if __name__ == '__main__':

    # Data
    (tr_en, tr_de, va_en, va_de,
     te_en, te_de, en_vocab, de_vocab) = load_data()
    en2de, de2en = make_loaders(tr_en, tr_de, va_en, va_de,
                                te_en, te_de, en_vocab, de_vocab)

    # Verify
    verify_sharing()

    # Dimensions
    dev = CONFIG['device']
    E   = CONFIG['embed_dim']
    H   = CONFIG['hidden_dim']
    D   = CONFIG['dropout']

    # ── BUILD ────────────────────────────────────────────────
    model_en2de  = Exp1Seq2Seq(len(en_vocab), len(de_vocab), E, H, D).to(dev)
    model_de2en  = Exp1Seq2Seq(len(de_vocab), len(en_vocab), E, H, D).to(dev)
    model_shared = Exp2SharedSeq2Seq(len(en_vocab), len(de_vocab), E, H, D).to(dev)

    # ── EXP 1a: EN→DE ────────────────────────────────────────
    model_en2de = train_exp1(model_en2de,
                             en2de['train'], en2de['val'],
                             "Exp1a: EN→DE  (separate enc/dec cells)")

    # ── EXP 1b: DE→EN ────────────────────────────────────────
    model_de2en = train_exp1(model_de2en,
                             de2en['train'], de2en['val'],
                             "Exp1b: DE→EN  (separate enc/dec cells)")

    # ── EXP 2: SHARED ────────────────────────────────────────
    model_shared = train_exp2(model_shared, en2de, de2en,
                              "Exp2:  EN↔DE  (one shared LSTM cell)")

    # ── BLEU ─────────────────────────────────────────────────
    print(f"\n{'='*55}\nBLEU Evaluation (Test Set)\n{'='*55}")

    b1_en2de = compute_bleu(model_en2de,  en2de['test'], de_vocab, dev)
    b1_de2en = compute_bleu(model_de2en,  de2en['test'], en_vocab, dev)
    b2_en2de = compute_bleu(model_shared, en2de['test'], de_vocab, dev, direction='en2de')
    b2_de2en = compute_bleu(model_shared, de2en['test'], en_vocab, dev, direction='de2en')

    p1 = count_params(model_en2de) + count_params(model_de2en)
    p2 = count_params(model_shared)

    # LSTM cell params only
    lstm_p1 = (sum(p.numel() for p in model_en2de.enc_cell.parameters()) +
               sum(p.numel() for p in model_en2de.dec_cell.parameters()) +
               sum(p.numel() for p in model_de2en.enc_cell.parameters()) +
               sum(p.numel() for p in model_de2en.dec_cell.parameters()))
    lstm_p2 = sum(p.numel() for p in model_shared.cell.parameters())

    print(f"\n{'Metric':<38} {'Exp1 (Separate)':>16} {'Exp2 (Shared)':>14}")
    print("-" * 70)
    print(f"{'EN→DE BLEU':<38} {b1_en2de:>16.2f} {b2_en2de:>14.2f}")
    print(f"{'DE→EN BLEU':<38} {b1_de2en:>16.2f} {b2_de2en:>14.2f}")
    print(f"{'Avg BLEU':<38} {(b1_en2de+b1_de2en)/2:>16.2f} {(b2_en2de+b2_de2en)/2:>14.2f}")
    print(f"{'Total params (both directions)':<38} {p1:>16,} {p2:>14,}")
    print(f"{'Total param reduction':<38} {'baseline':>16} {(1-p2/p1)*100:>13.1f}%")
    print(f"{'LSTM cell params only':<38} {lstm_p1:>16,} {lstm_p2:>14,}")
    print(f"{'LSTM cell reduction':<38} {'baseline':>16} {(1-lstm_p2/lstm_p1)*100:>13.1f}%")
    print(f"{'LSTM cell count':<38} {'4 cells':>16} {'1 cell':>14}")
    print(f"{'Weight sharing confirmed':<38} {'False':>16} {'True':>14}")

    # ── SAMPLE TRANSLATIONS ──────────────────────────────────
    samples = te_en[:5]
    show_translations(model_en2de, samples, en_vocab, de_vocab, dev,
                      label="Exp1a EN→DE")
    show_translations(model_shared, samples, en_vocab, de_vocab, dev,
                      direction='en2de', label="Exp2 EN→DE (shared cell)")

    de_samples = te_de[:5]
    show_translations(model_de2en, de_samples, de_vocab, en_vocab, dev,
                      label="Exp1b DE→EN")
    show_translations(model_shared, de_samples, de_vocab, en_vocab, dev,
                      direction='de2en', label="Exp2 DE→EN (shared cell)")

    # ── SAVE ─────────────────────────────────────────────────
    results = {
        'exp1_en2de_bleu': round(b1_en2de, 4),
        'exp1_de2en_bleu': round(b1_de2en, 4),
        'exp1_avg_bleu':   round((b1_en2de + b1_de2en) / 2, 4),
        'exp2_en2de_bleu': round(b2_en2de, 4),
        'exp2_de2en_bleu': round(b2_de2en, 4),
        'exp2_avg_bleu':   round((b2_en2de + b2_de2en) / 2, 4),
        'params_exp1_combined': p1,
        'params_exp2':          p2,
        'total_reduction_pct':  round((1 - p2 / p1) * 100, 2),
        'lstm_params_exp1':     lstm_p1,
        'lstm_params_exp2':     lstm_p2,
        'lstm_reduction_pct':   round((1 - lstm_p2 / lstm_p1) * 100, 2),
        'lstm_cells_exp1':      4,
        'lstm_cells_exp2':      1,
    }
    with open(CONFIG['save_path'], 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {CONFIG['save_path']}")

Device: cuda

Loading Multi30k from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train 29000 | Val 1014 | Test 1000
EN vocab 7,704 | DE vocab 9,597
EN→DE train batches: 227 | DE→EN train batches: 227

── Weight Sharing Verification ──
  Exp1 enc_cell IS dec_cell          : False  ✓ (should be False)
  Exp2 LSTMCell count in model       : 1  ✓ (should be 1)
  Exp2 cell id (encoder = decoder)   : 138135905072480  ✓ (same object all paths)

Training : Exp1a: EN→DE  (separate enc/dec cells)
Params   : 12,502,141
Epoch 01 | train 5.4042 | val 5.2276 | 31.8s
Epoch 02 | train 4.6669 | val 4.8250 | 31.9s
Epoch 03 | train 4.2212 | val 4.4990 | 32.6s
Epoch 04 | train 3.8882 | val 4.3449 | 32.0s
Epoch 05 | train 3.6612 | val 4.1409 | 31.9s
Epoch 06 | train 3.4565 | val 4.0470 | 31.9s
Epoch 07 | train 3.2714 | val 3.9355 | 31.9s
Epoch 08 | train 3.0976 | val 3.8794 | 32.0s
Epoch 09 | train 2.9649 | val 3.8155 | 31.9s
Epoch 10 | train 2.8260 | val 3.8110 | 32.1s
Epoch 11 | train 2.7125 | val 3.7324 | 32.5s
Epoch 12 | train 2.5993 | val 3.7378 | 32.2s
Epoch 13 | train 2.4841 | v

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/nmt_results.json'

In [3]:
"""
True B-BP Seq2Seq NMT — Adigun-Kosko Framework  (v3 — memory safe)
===================================================================
Fixes from v2:
  - Vocab capped at MAX_VOCAB=10000 (was 535k/888k → OOM)
  - Training data subsampled to 200k sentences (wmt14 is 4.5M → overkill)
  - Batch reduced to 64
  - hid_dim kept at 256

Install (run once in Colab):
  !pip install -q datasets sacrebleu
"""

import math, time, collections, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import sacrebleu

# ── 1. Device ─────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── 2. Hyperparameters ────────────────────────────────────────────────────────
MAX_VOCAB   = 10_000     # cap both EN and DE vocabularies
MAX_LEN     = 40         # max tokens per sentence
MIN_FREQ    = 2
BATCH       = 64
HID         = 256
LR          = 3e-4
EPOCHS      = 15
LAM         = 0.3        # cycle consistency weight
TRAIN_CAP   = 200_000    # subsample wmt14 to this many sentence pairs
SEED        = 42
random.seed(SEED); torch.manual_seed(SEED)

# ── 3. Data ───────────────────────────────────────────────────────────────────
print("Loading dataset (wmt14 en-de)...")
ds = load_dataset("wmt14", "de-en")
train_raw = ds["train"]
val_raw   = ds["validation"]
test_raw  = ds["test"]
print(f"Raw  train={len(train_raw):,}  val={len(val_raw):,}  test={len(test_raw):,}")

def get_pair(sample):
    t = sample.get("translation", {})
    return t.get("en", "").strip(), t.get("de", "").strip()

def simple_tok(text):
    return text.lower().split()

# ── 4. Vocabulary ─────────────────────────────────────────────────────────────
PAD, UNK, SOS, EOS = "<pad>", "<unk>", "<sos>", "<eos>"
SPECIALS = [PAD, UNK, SOS, EOS]

class Vocab:
    def __init__(self):
        self.itos = list(SPECIALS)
        self.stoi = {s: i for i, s in enumerate(SPECIALS)}

    def build(self, token_lists, min_freq, max_vocab):
        counter = collections.Counter(t for tl in token_lists for t in tl)
        # Take top (max_vocab - len(SPECIALS)) by frequency
        limit = max_vocab - len(SPECIALS)
        for word, freq in counter.most_common(limit):
            if freq < min_freq:
                break
            self.stoi[word] = len(self.itos)
            self.itos.append(word)

    def encode(self, tokens):
        unk = self.stoi[UNK]
        return [self.stoi.get(t, unk) for t in tokens]

    def __len__(self):
        return len(self.itos)

print(f"Building vocabularies from first {TRAIN_CAP:,} training pairs...")
# Subsample indices
all_indices = list(range(len(train_raw)))
random.shuffle(all_indices)
sub_indices = all_indices[:TRAIN_CAP]

en_tok_lists, de_tok_lists = [], []
valid_sub = []
for idx in sub_indices:
    en, de = get_pair(train_raw[idx])
    et, dt = simple_tok(en), simple_tok(de)
    if 1 <= len(et) <= MAX_LEN and 1 <= len(dt) <= MAX_LEN:
        en_tok_lists.append(et)
        de_tok_lists.append(dt)
        valid_sub.append(idx)

vocab_en = Vocab(); vocab_en.build(en_tok_lists, MIN_FREQ, MAX_VOCAB)
vocab_de = Vocab(); vocab_de.build(de_tok_lists, MIN_FREQ, MAX_VOCAB)
print(f"EN vocab: {len(vocab_en):,}  |  DE vocab: {len(vocab_de):,}")

PAD_IDX_EN = vocab_en.stoi[PAD]; SOS_IDX_EN = vocab_en.stoi[SOS]; EOS_IDX_EN = vocab_en.stoi[EOS]
PAD_IDX_DE = vocab_de.stoi[PAD]; SOS_IDX_DE = vocab_de.stoi[SOS]; EOS_IDX_DE = vocab_de.stoi[EOS]
SPEC_EN = {PAD_IDX_EN, SOS_IDX_EN, EOS_IDX_EN}
SPEC_DE = {PAD_IDX_DE, SOS_IDX_DE, EOS_IDX_DE}

# ── 5. Dataset encoding ───────────────────────────────────────────────────────
def encode_pair(en_str, de_str):
    et, dt = simple_tok(en_str), simple_tok(de_str)
    if not (1 <= len(et) <= MAX_LEN and 1 <= len(dt) <= MAX_LEN):
        return None
    en_ids = torch.tensor([SOS_IDX_EN] + vocab_en.encode(et) + [EOS_IDX_EN], dtype=torch.long)
    de_ids = torch.tensor([SOS_IDX_DE] + vocab_de.encode(dt) + [EOS_IDX_DE], dtype=torch.long)
    return en_ids, de_ids

def make_pairs_from_raw(raw_split, cap=None):
    pairs = []
    indices = list(range(len(raw_split)))
    if cap:
        random.shuffle(indices)
        indices = indices[:cap]
    for idx in indices:
        enc = encode_pair(*get_pair(raw_split[idx]))
        if enc:
            pairs.append(enc)
    return pairs

def make_pairs_from_lists(en_lists, de_lists, sub_idx):
    """Use already-tokenised lists to avoid re-iterating raw dataset."""
    pairs = []
    for et, dt in zip(en_lists, de_lists):
        en_ids = torch.tensor([SOS_IDX_EN] + vocab_en.encode(et) + [EOS_IDX_EN], dtype=torch.long)
        de_ids = torch.tensor([SOS_IDX_DE] + vocab_de.encode(dt) + [EOS_IDX_DE], dtype=torch.long)
        pairs.append((en_ids, de_ids))
    return pairs

def collate(batch):
    en_seqs, de_seqs = zip(*batch)
    return (pad_sequence(en_seqs, padding_value=PAD_IDX_EN),
            pad_sequence(de_seqs, padding_value=PAD_IDX_DE))

print("Encoding pairs...")
train_pairs = make_pairs_from_lists(en_tok_lists, de_tok_lists, valid_sub)
val_pairs   = make_pairs_from_raw(val_raw)
test_pairs  = make_pairs_from_raw(test_raw)
print(f"Train: {len(train_pairs):,}  Val: {len(val_pairs):,}  Test: {len(test_pairs):,}")

train_loader = DataLoader(train_pairs, batch_size=BATCH, shuffle=True,  collate_fn=collate, drop_last=True)
val_loader   = DataLoader(val_pairs,   batch_size=BATCH, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_pairs,  batch_size=BATCH, shuffle=False, collate_fn=collate)

# ── 6. B-BP RNN Cell ──────────────────────────────────────────────────────────
class BBPRNNCell(nn.Module):
    """
    One matrix U. Two directions via transpose.

    Forward  (bbp=False): h_t = tanh( h_{t-1} @ U   + x_t + b )
    Backward (bbp=True):  h_t = tanh( h_{t-1} @ U^T + x_t + b )

    This is the Adigun-Kosko inversion: same W used as W^T in reverse path.
    """
    def __init__(self, dim):
        super().__init__()
        k = 1.0 / math.sqrt(dim)
        self.U = nn.Parameter(torch.empty(dim, dim).uniform_(-k, k))
        self.b = nn.Parameter(torch.zeros(dim))

    def forward(self, x, h, bbp=False):
        W = self.U.T if bbp else self.U
        return torch.tanh(h @ W + x + self.b)

# ── 7. B-BP Seq2Seq ───────────────────────────────────────────────────────────
class BBPSeq2Seq(nn.Module):
    def __init__(self, en_vocab_size, de_vocab_size, hid_dim, dropout=0.3):
        super().__init__()
        self.hid_dim = hid_dim
        self.cell    = BBPRNNCell(hid_dim)
        self.E_en    = nn.Embedding(en_vocab_size, hid_dim, padding_idx=PAD_IDX_EN)
        self.E_de    = nn.Embedding(de_vocab_size, hid_dim, padding_idx=PAD_IDX_DE)
        self.drop    = nn.Dropout(dropout)
        self.b_enc   = nn.Parameter(torch.zeros(hid_dim))
        self.b_dec   = nn.Parameter(torch.zeros(hid_dim))

    def _enc(self, x, h, bbp):
        W = self.cell.U.T if bbp else self.cell.U
        return torch.tanh(h @ W + x + self.b_enc)

    def _dec(self, x, h, bbp):
        W = self.cell.U.T if bbp else self.cell.U
        return torch.tanh(h @ W + x + self.b_dec)

    def encode(self, src, emb, bbp):
        T, B = src.shape
        h = torch.zeros(B, self.hid_dim, device=src.device)
        for t in range(T):
            h = self._enc(self.drop(emb(src[t])), h, bbp)
        return h

    def decode(self, tgt_in, h, out_emb, bbp):
        T, B = tgt_in.shape
        logits = []
        for t in range(T):
            h = self._dec(self.drop(out_emb(tgt_in[t])), h, bbp)
            logits.append(self.drop(h) @ out_emb.weight.T)
        return torch.stack(logits)  # (T, B, V)

    def forward_en_de(self, src_en, tgt_de_in):
        return self.decode(tgt_de_in, self.encode(src_en, self.E_en, False), self.E_de, False)

    def forward_de_en(self, src_de, tgt_en_in):
        return self.decode(tgt_en_in, self.encode(src_de, self.E_de, True),  self.E_en, True)

    def cycle_en(self, src_en, tgt_en_in):
        """EN → encode(U) → decode(U^T) → EN_recon"""
        return self.decode(tgt_en_in, self.encode(src_en, self.E_en, False), self.E_en, True)

    def cycle_de(self, src_de, tgt_de_in):
        """DE → encode(U^T) → decode(U) → DE_recon"""
        return self.decode(tgt_de_in, self.encode(src_de, self.E_de, True),  self.E_de, False)

    @torch.no_grad()
    def translate(self, src, src_emb, out_emb, sos_idx, eos_idx, bbp, max_len=50):
        self.eval()
        h = self.encode(src, src_emb, bbp)
        B = src.shape[1]
        tok = torch.full((B,), sos_idx, dtype=torch.long, device=src.device)
        out = []
        for _ in range(max_len):
            h = self._dec(self.drop(out_emb(tok)), h, bbp)
            tok = (h @ out_emb.weight.T).argmax(-1)
            out.append(tok)
            if (tok == eos_idx).all():
                break
        return torch.stack(out).T  # (B, out_len)

    def translate_en_de(self, src): return self.translate(src, self.E_en, self.E_de, SOS_IDX_DE, EOS_IDX_DE, False)
    def translate_de_en(self, src): return self.translate(src, self.E_de, self.E_en, SOS_IDX_EN, EOS_IDX_EN, True)

# ── 8. Loss & BLEU ────────────────────────────────────────────────────────────
def xloss(logits, tgt, pad):
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1), ignore_index=pad)

def ids_to_str(ids, itos, spec):
    toks = []
    for t in ids:
        if t in spec: continue
        toks.append(itos[t])
    return " ".join(toks)

def compute_bleu(model, loader, direction, max_batches=30):
    model.eval()
    hyps, refs = [], []
    with torch.no_grad():
        for i, (en_b, de_b) in enumerate(loader):
            if i >= max_batches: break
            en_b, de_b = en_b.to(device), de_b.to(device)
            if direction == "en_de":
                preds   = model.translate_en_de(en_b)   # (B, T)
                ref_ids = de_b.T
                itos_h, itos_r, spec_h, spec_r = vocab_de.itos, vocab_de.itos, SPEC_DE, SPEC_DE
            else:
                preds   = model.translate_de_en(de_b)
                ref_ids = en_b.T
                itos_h, itos_r, spec_h, spec_r = vocab_en.itos, vocab_en.itos, SPEC_EN, SPEC_EN
            for p, r in zip(preds.tolist(), ref_ids.tolist()):
                hyps.append(ids_to_str(p,  itos_h, spec_h))
                refs.append([ids_to_str(r, itos_r, spec_r)])
    return sacrebleu.corpus_bleu(hyps, list(zip(*refs))).score

# ── 9. Train / eval loops ─────────────────────────────────────────────────────
def train_epoch(model, loader, opt, lam):
    model.train()
    tot = nmt_tot = cyc_tot = n = 0
    for en_b, de_b in loader:
        en_b, de_b = en_b.to(device), de_b.to(device)
        opt.zero_grad()

        l1 = xloss(model.forward_en_de(en_b, de_b[:-1]), de_b[1:], PAD_IDX_DE)
        l2 = xloss(model.forward_de_en(de_b, en_b[:-1]), en_b[1:], PAD_IDX_EN)
        l3 = xloss(model.cycle_en(en_b, en_b[:-1]),      en_b[1:], PAD_IDX_EN)
        l4 = xloss(model.cycle_de(de_b, de_b[:-1]),      de_b[1:], PAD_IDX_DE)

        nmt = l1 + l2
        cyc = l3 + l4
        loss = nmt + lam * cyc
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        tot += loss.item(); nmt_tot += nmt.item(); cyc_tot += cyc.item(); n += 1

    return tot/n, nmt_tot/n, cyc_tot/n

@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    tot, n = 0., 0
    for en_b, de_b in loader:
        en_b, de_b = en_b.to(device), de_b.to(device)
        l1 = xloss(model.forward_en_de(en_b, de_b[:-1]), de_b[1:], PAD_IDX_DE)
        l2 = xloss(model.forward_de_en(de_b, en_b[:-1]), en_b[1:], PAD_IDX_EN)
        tot += (l1+l2).item(); n += 1
    return tot/n

# ── 10. Main ──────────────────────────────────────────────────────────────────
def main():
    model = BBPSeq2Seq(len(vocab_en), len(vocab_de), HID).to(device)

    total    = sum(p.numel() for p in model.parameters())
    emb_params = model.E_en.weight.numel() + model.E_de.weight.numel()
    rec_params = model.cell.U.numel()

    print(f"\n{'='*65}")
    print(f"  B-BP NMT  |  Adigun-Kosko Bidirectional Backpropagation")
    print(f"{'='*65}")
    print(f"  Total params        : {total:,}")
    print(f"  Recurrent (U)       : {rec_params:,}  ← used as U and U^T")
    print(f"  Embeddings (tied)   : {emb_params:,}")
    print(f"  hid={HID}  batch={BATCH}  lr={LR}  λ={LAM}  epochs={EPOCHS}")

    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)

    best_val = 1e9
    print(f"\n{'Ep':>3} | {'Train':>8} | {'NMT':>8} | {'Cyc':>8} | "
          f"{'Val':>8} | {'EN→DE':>7} | {'DE→EN':>7} | {'s':>5}")
    print("-"*70)

    for ep in range(1, EPOCHS+1):
        t0 = time.time()
        tr, nt, cy = train_epoch(model, train_loader, opt, LAM)
        vl = eval_loss(model, val_loader)
        sched.step(vl)
        b1 = compute_bleu(model, val_loader, "en_de")
        b2 = compute_bleu(model, val_loader, "de_en")
        mark = " ◀" if vl < best_val else ""
        print(f"{ep:>3} | {tr:>8.4f} | {nt:>8.4f} | {cy:>8.4f} | "
              f"{vl:>8.4f} | {b1:>7.2f} | {b2:>7.2f} | {time.time()-t0:>5.0f}{mark}")
        if vl < best_val:
            best_val = vl
            torch.save(model.state_dict(), "bbp_best.pt")

    # ── Test ──────────────────────────────────────────────────────────────────
    print("\nLoading best checkpoint for test evaluation...")
    model.load_state_dict(torch.load("bbp_best.pt"))
    t1 = compute_bleu(model, test_loader, "en_de", max_batches=999)
    t2 = compute_bleu(model, test_loader, "de_en", max_batches=999)

    print(f"\n{'='*65}")
    print(f"  FINAL TEST RESULTS")
    print(f"{'='*65}")
    print(f"  BLEU EN→DE  (forward,  uses U)   : {t1:.2f}")
    print(f"  BLEU DE→EN  (backward, uses U^T) : {t2:.2f}")
    print(f"  Average BLEU                     : {(t1+t2)/2:.2f}")
    print(f"{'='*65}")

    # ── Sample translations ───────────────────────────────────────────────────
    print("\nSample translations:\n")
    model.eval()
    with torch.no_grad():
        for i, (en_b, de_b) in enumerate(test_loader):
            if i >= 3: break
            en1, de1 = en_b[:, :1].to(device), de_b[:, :1].to(device)
            src  = ids_to_str(en1.squeeze(1).tolist(), vocab_en.itos, SPEC_EN)
            rde  = ids_to_str(de1.squeeze(1).tolist(), vocab_de.itos, SPEC_DE)
            hde  = ids_to_str(model.translate_en_de(en1)[0].tolist(), vocab_de.itos, SPEC_DE)
            hen  = ids_to_str(model.translate_de_en(de1)[0].tolist(), vocab_en.itos, SPEC_EN)
            print(f"[EN src]   {src}")
            print(f"[DE ref]   {rde}")
            print(f"[DE hyp]   {hde}  ← forward  (U)")
            print(f"[EN back]  {hen}  ← backward (U^T)")
            print()

    print("Done. Checkpoint: bbp_best.pt")

if __name__ == "__main__":
    main()

Device: cuda
Loading dataset (wmt14 en-de)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

de-en/train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

...

de-en/test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

Raw  train=4,508,785  val=3,000  test=3,003
Building vocabularies from first 200,000 training pairs...
EN vocab: 10,000  |  DE vocab: 10,000
Encoding pairs...
Train: 176,513  Val: 2,839  Test: 2,874

  B-BP NMT  |  Adigun-Kosko Bidirectional Backpropagation
  Total params        : 5,186,304
  Recurrent (U)       : 65,536  ← used as U and U^T
  Embeddings (tied)   : 5,120,000
  hid=256  batch=64  lr=0.0003  λ=0.3  epochs=15

 Ep |    Train |      NMT |      Cyc |      Val |   EN→DE |   DE→EN |     s
----------------------------------------------------------------------
  1 |  85.0508 |  65.4407 |  65.3668 |  32.5057 |    3.95 |    0.00 |   560 ◀
  2 |  62.8019 |  48.3026 |  48.3312 |  32.2615 |    3.95 |    0.04 |   556 ◀
  3 |  55.6692 |  42.7972 |  42.9070 |  36.3973 |    3.95 |    2.70 |   558
  4 |  49.3893 |  38.0056 |  37.9456 |  29.4296 |    3.95 |    2.97 |   558 ◀
  5 |  44.0679 |  33.8910 |  33.9232 |  27.2674 |    3.95 |    2.63 |   556 ◀
  6 |  40.5086 |  31.1501 |  31.1950 